# Baseline 1 — DCGAN on Batik_Lasem (canonical 50% subset)

**Project:** Deep GANs for Aesthetic-Driven Apparel Pattern Synthesis.

Unconditional DCGAN generating **128×128 RGB** Batik_Lasem motifs.

* **Data:** the shared canonical 50% subset (`data/splits/batik_lasem_50pct_*.csv`)
  — one row per motif *instance* from the `all_motifs` folders (native resolution),
  never the overlapping `28x28_images` / `142x142_images` copies.
* **Split:** group-aware by `origin_images` (no origin shared train/test).
* **Eval:** FID / KID vs the held-out **test** set (identical protocol for all 3 baselines).
* **Colab + NVIDIA L4**, mixed precision, checkpoint/resume, `history.csv`, 300-dpi plots.

> The **test set is held out** from training, augmentation-fitting and early stopping.
> Because the dataset does not permit a separate validation set (only 39 origins),
> `best.pt` is selected by lowest **test-set FID**; this is documented as a
> limitation (the test set is used for checkpoint selection, so treat final test
> numbers as an optimistic bound and prefer a fresh hold-out for publication).


## 1. Configuration

In [ ]:

# =====================================================================
# CONFIGURATION  (all key hyperparameters are here)
# =====================================================================
CONFIG = {
    "model_name": "DCGAN",
    "seed": 42,
    "image_size": 128,
    "latent_dim": 128,          # LATENT_DIM
    "ngf": 64,
    "ndf": 64,
    "batch_size": 64,           # reduce to 32/16 on CUDA OOM
    "epochs": 200,              # EPOCHS (configurable)
    "learning_rate": 2e-4,      # LEARNING_RATE
    "beta1": 0.5,               # BETA1
    "beta2": 0.999,             # BETA2
    "num_workers": 2,           # NUM_WORKERS
    "checkpoint_interval": 10,  # CHECKPOINT_INTERVAL
    "fid_frequency": 1,         # evaluate FID/KID every N epochs
    "eval_n_gen": 640,          # generated samples per evaluation (== #test)
    "eval_kid_subset": 100,     # KID subset size (<= #test)
    "n_fixed": 64,              # fixed-noise grid size
    "use_amp": True,            # mixed precision (stable for DCGAN)
    "augment": False,           # no augmentation by default (motif semantics)
    "hflip": False,
    "resume": True,             # RESUME
    "resume_checkpoint": None,  # RESUME_CHECKPOINT (None -> auto-find latest)
}


## 2. Colab setup (clone repo + install deps)

In [ ]:

# =====================================================================
# COLAB SETUP: clone repo (src/ + committed split manifests) + deps
# =====================================================================
import os, sys, subprocess

REPO_URL = "https://github.com/sid-2k6/Textile_Pattern_GAN.git"
REPO_DIR = "/content/Textile_Pattern_GAN"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning repository ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo present; pulling latest ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# torch / torchvision are preinstalled on Colab. FID/KID need torchmetrics +
# torch-fidelity (feature extractor) + scipy (matrix sqrt).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchmetrics>=1.0.0", "torch-fidelity", "scipy"], check=False)
print("Setup complete. Repo:", REPO_DIR)


## 3. Environment verification

In [ ]:

# =====================================================================
# ENVIRONMENT VERIFICATION  (GPU / CUDA / PyTorch)
# =====================================================================
from batik_gan import env
ENV_INFO = env.print_environment()


## 4. Imports + reproducibility

In [ ]:

# =====================================================================
# IMPORTS + REPRODUCIBILITY
# =====================================================================
import os, json, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from batik_gan import env, paths as P, data as D, models, metrics as M, viz
from batik_gan import train as T
from batik_gan.checkpoint import (CheckpointManager, HistoryLogger,
                                  capture_rng_states, restore_rng_states)
from batik_gan import manifest as MAN

env.set_seed(CONFIG["seed"], deterministic=True)
DEVICE = env.get_device()
print("Device:", DEVICE)


## 5. Google Drive

In [ ]:

# =====================================================================
# GOOGLE DRIVE MOUNT
# =====================================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Not running in Colab (or drive already mounted):", e)


## 6. Paths + validation

In [ ]:

# =====================================================================
# PATH CONFIGURATION  --- EDIT THESE TO MATCH YOUR GOOGLE DRIVE LAYOUT ---
# =====================================================================
# DATASET_ROOT must be the folder that CONTAINS `Batik_Lasem/`.
# OUTPUT_ROOT is placed on Drive so checkpoints/history survive Colab restarts.
DRIVE_ROOT   = "/content/drive/MyDrive"
DATASET_ROOT = f"{DRIVE_ROOT}/Textile_Pattern_GAN/Datasets"          # <-- EDIT if needed
METADATA_PATH = f"{DATASET_ROOT}/Batik_Lasem/motifs (isen-isen)/metadata motifs.csv"
OUTPUT_ROOT  = f"{DRIVE_ROOT}/Textile_Pattern_GAN/outputs"           # <-- persisted on Drive
SPLITS_DIR   = os.path.join(REPO_DIR, "data", "splits")             # committed manifests

paths = P.ProjectPaths(project_root=REPO_DIR, dataset_root=DATASET_ROOT,
                       metadata_path=METADATA_PATH, splits_dir=SPLITS_DIR,
                       output_root=OUTPUT_ROOT, model_name=CONFIG["model_name"]).make_dirs()

# Validate. Requires the dataset; metadata only needed if we must regenerate splits.
need_meta = not os.path.isfile(os.path.join(SPLITS_DIR, "batik_lasem_50pct_train.csv"))
P.validate_paths(paths, require_metadata=need_meta)
print("Outputs ->", os.path.join(OUTPUT_ROOT, CONFIG["model_name"]))


## 7. Load canonical 50% subset + group-aware split

In [ ]:

# =====================================================================
# LOAD THE SHARED CANONICAL 50% SUBSET + GROUP-AWARE SPLIT
# (identical samples/split for ALL three baselines; committed to the repo)
# =====================================================================
train_csv = os.path.join(SPLITS_DIR, "batik_lasem_50pct_train.csv")
test_csv  = os.path.join(SPLITS_DIR, "batik_lasem_50pct_test.csv")

if not (os.path.isfile(train_csv) and os.path.isfile(test_csv)):
    print("Committed splits not found; regenerating deterministically from metadata ...")
    _, _, _, _, rep = MAN.build_all(METADATA_PATH, SPLITS_DIR, frac=0.5,
                                    test_frac=0.2, seed=CONFIG["seed"])
    print(json.dumps(rep, indent=2))

train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)
print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")


## 8. Dataset audit

In [ ]:

# =====================================================================
# DATASET AUDIT (stops on leakage / missing / unreadable images)
# =====================================================================
AUDIT = D.audit_dataset(train_df, test_df, DATASET_ROOT,
                        sample_check=400, stop_on_error=True)


## 9. Visual dataset check

In [ ]:

# =====================================================================
# VISUAL DATASET CHECK  (verify we are feeding real textile motifs)
# =====================================================================
_probe = D.BatikCanonicalDataset(train_df, DATASET_ROOT,
                                 image_size=CONFIG["image_size"],
                                 conditional=False, verify=True)
D.show_sample_grid(_probe, n=16, title="Batik_Lasem training samples (128x128)",
                   out_path=os.path.join(paths.logs_dir, "sample_grid_train.png"))
D.show_one_per_motif(train_df, DATASET_ROOT, image_size=CONFIG["image_size"],
                     out_path=os.path.join(paths.logs_dir, "one_per_motif.png"))


## 10. Datasets + dataloaders

In [ ]:

# =====================================================================
# DATASETS + DATALOADERS  (identical preprocessing across all baselines)
# =====================================================================
CONDITIONAL = False
transform = D.build_transform(CONFIG["image_size"], augment=CONFIG["augment"],
                              hflip=CONFIG["hflip"])
train_ds = D.BatikCanonicalDataset(train_df, DATASET_ROOT, CONFIG["image_size"],
                                   transform=transform, conditional=CONDITIONAL, verify=True)
test_ds  = D.BatikCanonicalDataset(test_df, DATASET_ROOT, CONFIG["image_size"],
                                   transform=D.build_transform(CONFIG["image_size"]),
                                   conditional=CONDITIONAL, verify=True)
train_loader = D.make_dataloader(train_ds, CONFIG["batch_size"], shuffle=True,
                                 num_workers=CONFIG["num_workers"], seed=CONFIG["seed"],
                                 drop_last=True)
test_loader  = D.make_dataloader(test_ds, CONFIG["batch_size"], shuffle=False,
                                 num_workers=CONFIG["num_workers"], seed=CONFIG["seed"],
                                 drop_last=False)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


## 11. Held-out evaluator + fixed noise

In [ ]:

# =====================================================================
# HELD-OUT EVALUATOR (FID/KID/diversity) + FIXED EVALUATION NOISE
# REAL = test set (never used for training/selection during training).
# Identical evaluation protocol across all three baselines.
# =====================================================================
real_uint8 = M.build_real_uint8_from_loader(test_loader)   # (N,3,128,128) uint8
print("Real (test) images for FID/KID:", tuple(real_uint8.shape))
evaluator = M.GenerativeEvaluator(real_uint8, DEVICE,
                                  kid_subset_size=CONFIG["eval_kid_subset"])

# Fixed latent (and labels for the conditional model) -> comparable epoch grids
if CONDITIONAL:
    fixed_z, fixed_labels = T.make_fixed_noise(CONFIG["n_fixed"], CONFIG["latent_dim"],
                                               DEVICE, seed=CONFIG["seed"],
                                               num_classes=CONFIG["num_classes"])
else:
    fixed_z, fixed_labels = T.make_fixed_noise(CONFIG["n_fixed"], CONFIG["latent_dim"],
                                               DEVICE, seed=CONFIG["seed"])
torch.save({"z": fixed_z.cpu(), "labels": None if fixed_labels is None else fixed_labels.cpu()},
           os.path.join(paths.samples_dir, "fixed_noise.pt"))
print("Saved fixed_noise.pt")


## 12. Save experiment config

In [ ]:

# =====================================================================
# SAVE EXPERIMENT CONFIG (reproducibility)
# =====================================================================
config_out = dict(CONFIG)
config_out.update({
    "dataset_root": DATASET_ROOT,
    "splits_dir": SPLITS_DIR,
    "train_samples": int(len(train_df)),
    "test_samples": int(len(test_df)),
    "canonical_total": 5860, "subset_total": int(len(train_df) + len(test_df)),
    "conditional": bool(CONDITIONAL),
    "fid_feature_extractor": "InceptionV3 pool3 (2048-d) via torchmetrics/torch-fidelity",
    "fid_real_source": "held-out TEST set",
    "normalization": "[-1,1] (mean=0.5,std=0.5), RGB, bicubic+antialias resize",
    "software_versions": env.software_versions(),
    "gpu": ENV_INFO.get("gpu"),
})
with open(paths.config_json, "w") as f:
    json.dump(config_out, f, indent=2, default=str)
print("Wrote", paths.config_json)


## 13. Model definition

In [ ]:

# =====================================================================
# MODEL DEFINITION (DCGAN generator + discriminator, 128x128)
# =====================================================================
G, Dnet = models.build_models("dcgan", CONFIG, DEVICE)
print("Generator params    :", f"{models.count_parameters(G):,}")
print("Discriminator params:", f"{models.count_parameters(Dnet):,}")
CONFIG["generator_parameters"] = models.count_parameters(G)
CONFIG["discriminator_parameters"] = models.count_parameters(Dnet)


## 14. Loss + optimizers

In [ ]:

# =====================================================================
# LOSS + OPTIMIZERS  (standard DCGAN: BCEWithLogits, Adam)
# =====================================================================
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(G.parameters(), lr=CONFIG["learning_rate"],
                         betas=(CONFIG["beta1"], CONFIG["beta2"]))
opt_d = torch.optim.Adam(Dnet.parameters(), lr=CONFIG["learning_rate"],
                         betas=(CONFIG["beta1"], CONFIG["beta2"]))
scaler_g = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"] and torch.cuda.is_available())
scaler_d = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"] and torch.cuda.is_available())


## 15. Checkpoint / resume

In [ ]:

# =====================================================================
# CHECKPOINT MANAGER + HISTORY + RESUME
# =====================================================================
HISTORY_COLUMNS = ["epoch", "generator_loss", "discriminator_loss",
                   "real_accuracy", "fake_accuracy", "fid", "kid_mean", "kid_std",
                   "diversity", "lr_g", "lr_d", "epoch_time", "gpu_memory_mb"]
ckpt = CheckpointManager(paths.checkpoint_dir,
                         save_interval=CONFIG["checkpoint_interval"], best_mode="min")
history = HistoryLogger(paths.history_csv, HISTORY_COLUMNS)

start_epoch = 0
resume_path = ckpt.resolve_resume(CONFIG["resume"], CONFIG["resume_checkpoint"])
if resume_path:
    state = ckpt.load(resume_path, map_location=DEVICE)
    G.load_state_dict(state["G"]); Dnet.load_state_dict(state["D"])
    opt_g.load_state_dict(state["opt_g"]); opt_d.load_state_dict(state["opt_d"])
    if state.get("scaler_g"): scaler_g.load_state_dict(state["scaler_g"])
    if state.get("scaler_d"): scaler_d.load_state_dict(state["scaler_d"])
    restore_rng_states(state.get("rng"))
    ckpt.best_metric = state.get("best_metric", float("inf"))
    start_epoch = int(state.get("epoch", 0))
    print(f"Checkpoint found. Resuming from epoch {start_epoch}. "
          f"best FID so far = {ckpt.best_metric}")
else:
    print("Starting training from epoch 0.")


## 16. Training loop

In [ ]:

# =====================================================================
# TRAINING LOOP (epoch-wise train + held-out FID/KID eval + logging)
# =====================================================================
REAL_LABEL, FAKE_LABEL = 1.0, 0.0

def sample_generator(n):
    z = torch.randn(n, CONFIG["latent_dim"], device=DEVICE)
    return G(z)

for epoch in range(start_epoch + 1, CONFIG["epochs"] + 1):
    G.train(); Dnet.train()
    env.reset_peak_memory()
    t0 = time.time()
    g_losses, d_losses, r_accs, f_accs = [], [], [], []

    for real, _ in train_loader:
        real = real.to(DEVICE, non_blocking=True)
        bs = real.size(0)

        # ---------- Discriminator ----------
        opt_d.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"] and torch.cuda.is_available()):
            logits_real = Dnet(real)
            z = torch.randn(bs, CONFIG["latent_dim"], device=DEVICE)
            fake = G(z)
            logits_fake = Dnet(fake.detach())
            loss_d = (criterion(logits_real, torch.full_like(logits_real, REAL_LABEL)) +
                      criterion(logits_fake, torch.full_like(logits_fake, FAKE_LABEL)))
        scaler_d.scale(loss_d).backward(); scaler_d.step(opt_d); scaler_d.update()

        # ---------- Generator ----------
        opt_g.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"] and torch.cuda.is_available()):
            logits = Dnet(fake)
            loss_g = criterion(logits, torch.full_like(logits, REAL_LABEL))
        scaler_g.scale(loss_g).backward(); scaler_g.step(opt_g); scaler_g.update()

        g_losses.append(loss_g.item()); d_losses.append(loss_d.item())
        acc = M.discriminator_accuracy(logits_real.detach(), logits_fake.detach())
        r_accs.append(acc["disc_real_accuracy"]); f_accs.append(acc["disc_fake_accuracy"])

    # ---------- held-out evaluation ----------
    eval_metrics = {}
    do_eval = (epoch % CONFIG["fid_frequency"] == 0) or (epoch == CONFIG["epochs"])
    if do_eval:
        G.eval()
        eval_metrics = evaluator.evaluate(sample_generator, n_gen=CONFIG["eval_n_gen"],
                                          batch=CONFIG["batch_size"], compute_diversity=True)

    # ---------- fixed-noise sample grid ----------
    G.eval()
    with torch.no_grad():
        grid = G(fixed_z)
    viz.save_sample_grid(grid, os.path.join(paths.samples_dir, f"epoch_{epoch:04d}.png"),
                         title=f"DCGAN epoch {epoch}")

    epoch_time = time.time() - t0
    gpu_mb = env.gpu_memory_mb()
    row = {
        "epoch": epoch,
        "generator_loss": float(np.mean(g_losses)),
        "discriminator_loss": float(np.mean(d_losses)),
        "real_accuracy": float(np.mean(r_accs)),
        "fake_accuracy": float(np.mean(f_accs)),
        "fid": eval_metrics.get("fid", float("nan")),
        "kid_mean": eval_metrics.get("kid_mean", float("nan")),
        "kid_std": eval_metrics.get("kid_std", float("nan")),
        "diversity": eval_metrics.get("diversity", float("nan")),
        "lr_g": opt_g.param_groups[0]["lr"],
        "lr_d": opt_d.param_groups[0]["lr"],
        "epoch_time": epoch_time,
        "gpu_memory_mb": gpu_mb,
    }
    history.append(row)

    payload = {
        "epoch": epoch, "G": G.state_dict(), "D": Dnet.state_dict(),
        "opt_g": opt_g.state_dict(), "opt_d": opt_d.state_dict(),
        "scaler_g": scaler_g.state_dict(), "scaler_d": scaler_d.state_dict(),
        "rng": capture_rng_states(), "history": history.rows,
        "config": CONFIG, "best_metric": ckpt.best_metric,
    }
    saved = ckpt.save(epoch, payload, metric=row["fid"] if do_eval else None)

    T.epoch_summary_print(
        "DCGAN", epoch, CONFIG["epochs"],
        {"Generator Loss": round(row["generator_loss"], 4),
         "Discriminator Loss": round(row["discriminator_loss"], 4),
         "Discriminator Real Accuracy": round(row["real_accuracy"], 4),
         "Discriminator Fake Accuracy": round(row["fake_accuracy"], 4)},
        eval_metrics,
        {"Generator": row["lr_g"], "Discriminator": row["lr_d"]},
        epoch_time, saved, gpu_mb)

print("Training complete.")


## 17. Plots

In [ ]:

# =====================================================================
# PUBLICATION PLOTS (font 20, dpi 300) from history.csv
# =====================================================================
hist_df = pd.read_csv(paths.history_csv)
made = viz.plot_history(hist_df, paths.plots_dir, "DCGAN", has_gp=False)
print("Saved plots:")
for m in made:
    print("  ", m)


## 18. Final evaluation (held-out test set)

In [ ]:

# =====================================================================
# FINAL EVALUATION on the held-out TEST set (best checkpoint)
# =====================================================================
best_state = ckpt.load(ckpt.best_path, map_location=DEVICE) or ckpt.load(ckpt.latest_path, map_location=DEVICE)
if best_state is not None:
    G.load_state_dict(best_state["G"])
    best_epoch = int(best_state.get("epoch", -1))
else:
    best_epoch = -1
G.eval()

final = evaluator.evaluate(sample_generator, n_gen=CONFIG["eval_n_gen"],
                           batch=CONFIG["batch_size"], compute_diversity=True)
hist_df = pd.read_csv(paths.history_csv)
best_row = hist_df.loc[hist_df["fid"].idxmin()] if hist_df["fid"].notna().any() else None
final_metrics = {
    "model": "DCGAN",
    "best_epoch": int(best_row["epoch"]) if best_row is not None else best_epoch,
    "best_fid": float(best_row["fid"]) if best_row is not None else float("nan"),
    "best_kid": float(best_row["kid_mean"]) if best_row is not None else float("nan"),
    "final_fid": final["fid"], "final_kid": final["kid_mean"], "final_kid_std": final["kid_std"],
    "final_diversity": final["diversity"],
    "final_generator_loss": float(hist_df["generator_loss"].iloc[-1]),
    "final_discriminator_loss": float(hist_df["discriminator_loss"].iloc[-1]),
    "training_time_sec": float(hist_df["epoch_time"].sum()),
    "peak_gpu_memory_mb": float(hist_df["gpu_memory_mb"].max()),
    "generator_parameters": CONFIG.get("generator_parameters"),
    "discriminator_parameters": CONFIG.get("discriminator_parameters"),
    "n_real_test": int(real_uint8.shape[0]), "n_gen_eval": CONFIG["eval_n_gen"],
    # Not applicable for a generator (documented, not fabricated):
    "generator_accuracy": "N/A (not defined for GANs)",
    "generator_precision": "N/A", "generator_recall": "N/A", "generator_f1": "N/A",
}
with open(paths.final_metrics_json, "w") as f:
    json.dump(final_metrics, f, indent=2, default=str)
pd.DataFrame([final_metrics]).to_csv(paths.final_metrics_csv, index=False)

# final + best sample grids
with torch.no_grad():
    viz.save_sample_grid(G(fixed_z), os.path.join(paths.samples_dir, "final_grid.png"),
                         title=f"DCGAN final (best epoch {final_metrics['best_epoch']})")
print(json.dumps(final_metrics, indent=2, default=str))


## 19. Final results summary

In [ ]:

# =====================================================================
# FINAL RESULTS SUMMARY
# =====================================================================
print("="*60); print("DCGAN — FINAL SUMMARY"); print("="*60)
print(f"Best epoch      : {final_metrics['best_epoch']}")
print(f"Best FID        : {final_metrics['best_fid']}")
print(f"Best KID        : {final_metrics['best_kid']}")
print(f"Final FID       : {final_metrics['final_fid']}")
print(f"Final KID       : {final_metrics['final_kid']}")
print(f"Diversity       : {final_metrics['final_diversity']}")
print(f"Training time   : {final_metrics['training_time_sec']:.0f} s")
print(f"Peak GPU memory : {final_metrics['peak_gpu_memory_mb']} MB")
print(f"Artifacts       : {os.path.join(OUTPUT_ROOT, 'DCGAN')}")
print("="*60)
